# Result inspection notebook

This notebook only reads files under `results/`. It does not run detection or
invent values. Missing artifacts are shown as **Not evaluated**.

Run from the project root or from `notebooks/`; the path logic below handles both.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
print("project root:", ROOT)
print("result dir exists:", RESULTS.exists())

## 1. Dataset audit

In [ ]:
path = RESULTS / "metrics" / "dataset_validation.json"
if path.is_file():
    report = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for split, data in report.get("splits", {}).items():
        rows.append({"split": split, "images": data["images"], "boxes": data["boxes"], **data["class_counts"]})
    display(pd.DataFrame(rows))
    print("status:", report.get("status"), "| errors:", len(report.get("errors", [])))
else:
    print("Not evaluated - results/metrics/dataset_validation.json missing")

## 2. Fixed-time baseline vs emergency priority

The comparison is a deterministic signal-level simulation, not a road trial.

In [ ]:
path = RESULTS / "metrics" / "comparison.json"
if path.is_file():
    sim = json.loads(path.read_text(encoding="utf-8"))
    b, p = sim["baseline"], sim["proposed"]
    table = pd.DataFrame([
        ["Emergency mean wait (s)", b.get("emergency_waiting_time_s"), p.get("emergency_waiting_time_s")],
        ["Emergency max wait (s)", b.get("emergency_max_wait_s"), p.get("emergency_max_wait_s")],
        ["Normal mean wait (s)", b.get("normal_waiting_time_s"), p.get("normal_waiting_time_s")],
        ["Normal max wait (s)", b.get("normal_max_wait_s"), p.get("normal_max_wait_s")],
        ["Throughput (veh/h)", b.get("throughput_vph"), p.get("throughput_vph")],
        ["Priority activations", b.get("priority_events"), p.get("priority_events")],
    ], columns=["Metric", "Fixed-time", "Priority"])
    display(table)
    events = pd.DataFrame(sim.get("priority_events", []))
    display(events[["event_id", "vehicle_class", "lane", "arrival_time", "green_time", "waiting_time_s", "baseline_wait_s"]])
else:
    print("Not evaluated - run python simulation.py")

## 3. Arrival-rate sweep

In [ ]:
path = RESULTS / "simulation" / "density_sweep.csv"
if path.is_file():
    sweep = pd.read_csv(path)
    display(sweep)
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for density, group in sweep.groupby("density", sort=False):
        ax.scatter(group["baseline_emergency_wait_s"], group["proposed_emergency_wait_s"], label=density)
    ax.set_xlabel("Baseline EV wait (s)")
    ax.set_ylabel("Proposed EV wait (s)")
    ax.set_title("Density sweep: EV waiting time (actual runs)")
    ax.legend(title="density")
    ax.grid(alpha=0.25)
    plt.show()
else:
    print("Not evaluated - run python simulation.py --sweep")

## 4. Held-out detector metrics

Only present after `python scripts/evaluate.py`.

In [ ]:
path = RESULTS / "metrics" / "detection_metrics.json"
if path.is_file() and json.loads(path.read_text(encoding="utf-8")).get("status") == "evaluated":
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data["overall"], indent=2))
    display(pd.DataFrame(data.get("per_class", [])))
    op = data.get("operating_point", {})
    print("operating point:", {k: op.get(k) for k in ("conf_threshold", "tp", "fp", "fn", "precision", "recall", "f1")})
else:
    print("Not evaluated - model or evaluation missing")

## 5. Training curve

Reads the per-epoch CSV copied by the training script. The smoke run and the
final run are distinct; this notebook reads the file that was copied last.

In [ ]:
path = RESULTS / "training" / "results.csv"
if path.is_file():
    history = pd.read_csv(path)
    history.columns = [c.strip() for c in history.columns]
    display(history.tail())
    map_col = next((c for c in history.columns if "mAP50-95" in c), None)
    if map_col:
        ax = history.plot(x="epoch", y=["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", map_col], figsize=(8, 4))
        ax.set_ylabel("score")
        ax.set_title("Validation metrics by epoch")
        ax.grid(alpha=0.25)
        plt.show()
else:
    print("Not evaluated - no training history")

## 6. Reading the results honestly

- EV waiting time is measured from the first confident lane-assigned detection,
  not from a physical stop-line sensor.
- The baseline is a counterfactual fixed-time schedule, not a field-optimised plan.
- Some individual priority events are slower than the baseline because an EV
  arriving on green still pays confirmation + preparation. The mean can improve
  while individual events do not.
- Fire-engine test data is scarce (20 boxes). Treat its per-class metrics cautiously.
- The simulation is not a road-safety, emissions or response-time claim.